In [1]:
# import libraries

import pandas as pd
import json
from pathlib import Path

In [2]:
# Input and output folders (update these paths to your local folders)

# Input folder for StatsBomb data

data_folder = 'C:/Users/bquev/Desktop/6311 Capstone/Data/open-data-master/open-data-master/data/'

# Output folder for processed data

data_output_folder = 'C:/Users/bquev/Desktop/6311 Capstone/output/processed_data/'

## Flattening the JSON files

StatsBomb data is stored as nested JSON files. The function below loads the data and uses pd.json_normalize() to flatten it into a dataframe. Nested columns are separated by underscores, while the lists remain unchanged.

**References:**

- [Python json.load()](https://docs.python.org/3/library/json.html#json.load)
- [pandas json_normalize()](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html)

In [3]:
# Load a JSON file and flatten it into a pandas DataFrame

def load_flat_json(file_path):
    with open(file_path, encoding='utf-8') as file:
        data = json.load(file)
    return pd.json_normalize(data, sep='_')

## Competitions

In [4]:
# Load Competitions data and turn into a pandas dataframe

competitions_df = load_flat_json(data_folder + 'competitions.json')
competitions_df.head()

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,NaN,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,NaN,2024-02-13T02:35:28.134882


In [5]:
# Checking number of seasons (rows) and columns

competitions_df.shape

(80, 12)

In [6]:
# Checking number of unique competitions in dataframe

competitions_df['competition_id'].nunique()

24

In [7]:
# Checking number of seasons per competition

competitions_df[['competition_id', 'competition_name']].value_counts()

competition_id  competition_name       
16              Champions League           18
11              La Liga                    18
43              FIFA World Cup              8
37              FA Women's Super League     4
87              Copa del Rey                3
7               Ligue 1                     3
9               1. Bundesliga               2
81              Liga Profesional            2
49              NWSL                        2
2               Premier League              2
12              Serie A                     2
55              UEFA Euro                   2
53              UEFA Women's Euro           2
72              Women's World Cup           2
1267            African Cup of Nations      1
223             Copa America                1
1470            FIFA U20 World Cup          1
135             Frauen Bundesliga           1
1238            Indian Super league         1
182             Liga F                      1
44              Major League Soccer     

The competitions file contains 80 competition seasons across 24 competitions. This dataframe has an uneven number of seasons per competition.

## Matches

Each competition has its own folder inside `matches/`, with a JSON file for each season.

The path is set to the `matches/` folder inside the data folder. From there, `Path.rglob('*.json')` finds every JSON file in the folder and its subfolders. Each file is then loaded and flattened into a DataFrame using the `load_flat_json()` function from earlier and added to a list. Once all the files are loaded, `pd.concat()` combines them into one `matches_df`.

**References:**

* [Python documentation, pathlib Path.rglob](https://docs.python.org/3/library/pathlib.html#pathlib.Path.rglob)



In [8]:

# Load and flatten each season's matches data into a single dataframe

# Start with an empty list to hold each season's matches

matches_list = []

# Find every .json file inside matches and its competition folders, and load each .json

for file_path in Path(data_folder + 'matches').rglob('*.json'):
    matches_list.append(load_flat_json(file_path))   

# Make all .jsons into one matches dataframe
matches_df = pd.concat(matches_list, ignore_index=True)

# Checking if load worked

matches_df.shape


(3961, 42)

In [9]:
# Checking for duplicate matches

matches_df['match_id'].nunique()

3961

In [29]:
# Checking matches data

matches_df.head()

,match_id,match_date,kick_off,home_score,away_score,match_status,match_status_360,last_updated,last_updated_360,match_week,...,competition_stage_id,competition_stage_name,stadium_id,stadium_name,stadium_country_id,stadium_country_name,referee_id,referee_name,referee_country_id,referee_country_name
0,9880,2018-04-14,16:15:00.000,2,1,available,scheduled,2023-02-08T17:23:53.901920,2021-06-13T16:17:31.694,32,...,1,Regular Season,342.0,Spotify Camp Nou,214.0,Spain,2728.0,Carlos del Cerro Grande,214.0,Spain
1,9912,2018-04-29,20:45:00.000,2,4,available,scheduled,2022-12-05T14:42:44.641092,2021-06-13T16:17:31.694,35,...,1,Regular Season,4658.0,Estadio Abanca-Riazor,214.0,Spain,2602.0,Ricardo De Burgos Bengoetxea,214.0,Spain
2,9924,2018-05-06,20:45:00.000,2,2,available,scheduled,2022-12-01T03:25:12.063586,2021-06-13T16:17:31.694,36,...,1,Regular Season,342.0,Spotify Camp Nou,214.0,Spain,2608.0,Alejandro José Hernández Hernández,214.0,Spain
3,9855,2018-03-18,16:15:00.000,2,0,available,scheduled,2022-12-01T02:33:31.178193,2021-06-13T16:17:31.694,29,...,1,Regular Season,342.0,Spotify Camp Nou,214.0,Spain,2575.0,Santiago Jaime Latre,214.0,Spain
4,9827,2018-03-01,21:00:00.000,1,1,available,scheduled,2022-08-04T17:18:06.540844,2021-06-13T16:17:31.694,26,...,1,Regular Season,357.0,Estadio de Gran Canaria,214.0,Spain,180.0,Antonio Miguel Mateu Lahoz,214.0,Spain


## Events

The event .json files are stored directly in the events folder rather than in subfolders. Due to the large amount of data in each file, the code below will only keep shot events in the DataFrame.

In [10]:
# Load and flatten each events data, only keeping shots, into a single dataframe

shots_list = []

# Use every events .json file, flatten it, then only keep the shots from the type_name column

for file_path in Path(data_folder + 'events').glob('*.json'):
    events_df = load_flat_json(file_path)
    match_shots_df = events_df[events_df['type_name'] == 'Shot'].copy()

    # Since the .json file id is the match id, a match_id column must be made based off the .json name using .stem from pathlib
    match_shots_df['match_id'] = int(file_path.stem)
    shots_list.append(match_shots_df)

shots_df = pd.concat(shots_list, ignore_index=True)

shots_df.shape


(108281, 150)

In [11]:
# Checking number of match ids in events data

shots_df['match_id'].nunique()

4235

In [12]:
# Checking how many match ids in matches_df also appear in shots_df

matches_df['match_id'].isin(shots_df['match_id']).sum()

np.int64(3961)

### Shots without a match record

Checking which shots in the events data come from matches that are not in matches_df, since they can't be linked to competition or season information from the matches data.

In [13]:
# Using an isin() as false to find the mismatches

events_missing_matches_df = shots_df[shots_df['match_id'].isin(matches_df['match_id']) == False]

print('Missing matches: ', events_missing_matches_df['match_id'].nunique())

Missing matches:  274


In [16]:
# Remove shots from matches with no match record in matches_df

shots_df = shots_df[shots_df['match_id'].isin(matches_df['match_id'])].copy()
print(shots_df['match_id'].nunique())
print(shots_df.shape)

3961
(101227, 150)


In [18]:
shots_df.head()

,id,index,period,timestamp,minute,second,possession,duration,type_id,type_name,...,shot_redirect,goalkeeper_lost_out,player_off_permanent,goalkeeper_lost_in_play,goalkeeper_success_out,goalkeeper_success_in_play,goalkeeper_saved_to_post,half_end_early_video_end,shot_kick_off,goalkeeper_penalty_saved_to_post
0,becd7956-ce44-479e-8fc9-16a2d1f1f349,137,1,00:02:29.477,2,29,6,1.075902,16,Shot,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9107d374-2942-4876-a14f-1b9f86901c15,262,1,00:05:39.641,5,39,12,0.807592,16,Shot,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ddd194ca-08fb-43d0-87c2-33647f975f9f,715,1,00:15:29.059,15,29,23,0.979318,16,Shot,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,86596ddb-d824-4e5e-b18c-b4442e9ce7cf,743,1,00:16:20.072,16,20,30,0.312149,16,Shot,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3ed2b107-be17-42d5-9d1b-25006a0e55cb,802,1,00:18:16.362,18,16,33,0.937618,16,Shot,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Men's matches only

Since we are working with a club in a top European men's league, we are excluding women's competitions because attacking styles and scoring patterns may differ.

In [21]:
# Create a men's matches dataframe to filter shots_df, which has no gender column

male_competition_ids = competitions_df[competitions_df['competition_gender'] == 'male']['competition_id']
male_matches_df = matches_df[matches_df['competition_competition_id'].isin(male_competition_ids)].copy()
male_matches_df.shape

(2651, 42)

In [23]:
# Use male_matches_df to exclude women from shots_df

shots_df = shots_df[shots_df['match_id'].isin(male_matches_df['match_id'])].copy()

# Check data
print(shots_df.shape)
print(shots_df['match_id'].nunique())

(66854, 150)
2651


## Final Dataset to CSV

Merging match details into `shots_df` so everything needed for analysis is in one DataFrame.

**References:**

* [pandas DataFrame.merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)

In [ ]:
# Columns to add from matches_df, can be adjusted later

match_columns = ['match_id', 'match_date',
                 'competition_competition_id', 'competition_competition_name',
                 'season_season_id', 'season_season_name',
                 'home_team_home_team_name', 'away_team_away_team_name',
                 'home_score', 'away_score']

# Merging shots_df with matches_df, using a validation of many match ids to 1 in matches_df

shots_df = shots_df.merge(
    matches_df[match_columns],
    on = 'match_id',
    how = 'left',
    validate = 'many_to_one'
)

print(shots_df.shape)
print(shots_df['match_id'].nunique())

(66854, 159)
2651


In [27]:
# Rename the doubled column names from flattening to shorter names

shots_df = shots_df.rename(columns={'competition_competition_id': 'competition_id',
                                    'competition_competition_name': 'competition_name',
                                    'season_season_id': 'season_id',
                                    'season_season_name': 'season_name',
                                    'home_team_home_team_name': 'home_team',
                                    'away_team_away_team_name': 'away_team'})

shots_df.columns

Index(['id', 'index', 'period', 'timestamp', 'minute', 'second', 'possession',
       'duration', 'type_id', 'type_name',
       ...
       'goalkeeper_penalty_saved_to_post', 'match_date', 'competition_id',
       'competition_name', 'season_id', 'season_name', 'home_team',
       'away_team', 'home_score', 'away_score'],
      dtype='str', length=159)

## Final dataset to csv

In [ ]:
shots_df.to_csv(data_output_folder + 'shots.csv', index = False)

## Data Attribution

Data provided by StatsBomb: https://github.com/hudl/open-data

![StatsBomb logo](images/statsbomb_logo.png)